# Gün 2 (25 Ağustos) - DOC-25: Etiketlerin RAG pipeline'a meta veri olarak eklenmesi

DOC-24'te (notebook 08) her belge için `classify_document()` ile üretilen `siniflar` / `guven` / `etiketler` / `human_review` alanlarını, RAG pipeline'ının FAISS metadata'sına (her chunk'a) ekliyoruz.

**Uçtan uca zincir #2 (OCR → RAG → Sınıflandırma):** Aynı test belgeleri (test_talep_01-05) üzerinden OCR → RAG → sınıflandırma zincirinin kırılmadan çalıştığı; yani sınıflandırma meta verisi eklendikten sonra da DOC-23'teki arama doğruluğunun (Hit@3=%100) korunduğu ve her arama sonucunun artık kendi sınıf/etiket/human_review bilgisini taşıdığı doğrulanacak.

In [1]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from text_splitter import split_text
from embedder import embed_chunks
from vector_store import build_index, save_index, load_index, search, load_index_path
from classifier import classify_document, attach_labels_to_chunks, DEFAULT_CONFIDENCE_THRESHOLD

print("Moduller yuklendi.")

Moduller yuklendi.


## 1. DOC-23'te doğrulanmış gerçek OCR çıktılarını yükle

Rastgele metin yerine, `ocr_real_outputs.json` içindeki (DOC-16/17/18 pipeline'ından gelen ve DOC-23'te ground truth ile 25/25 alan birebir eşleştiği doğrulanmış) gerçek belge verisi kullanılıyor.

In [2]:
OCR_PATH = "../data/processed/ocr_real_outputs.json"
with open(OCR_PATH, encoding="utf-8") as f:
    ocr_outputs = json.load(f)


def format_document(fields: dict) -> str:
    return (
        f"Talep Eden: {fields['talep_eden']}\n"
        f"Tarih: {fields['tarih']}\n"
        f"Departman: {fields['departman']}\n"
        f"Konu: {fields['konu']}\n\n"
        f"{fields['aciklama']}"
    )


print(f"{len(ocr_outputs)} belge yuklendi: {list(ocr_outputs.keys())}")

5 belge yuklendi: ['test_talep_01.png', 'test_talep_02.png', 'test_talep_03.png', 'test_talep_04.png', 'test_talep_05.png']


## 2. Her belgeyi sınıflandır (DOC-24 zinciri)

`classify_document()` her belge için `{siniflar, guven, etiketler, gerekce, human_review}` döndürüyor. Sonuçlar `source_doc` (dosya adı) anahtarıyla bir sözlükte tutuluyor; bir sonraki adımda bu sözlük chunk'lara meta veri olarak eklenecek.

In [3]:
classifications = {}
for filename, fields in sorted(ocr_outputs.items()):
    result = classify_document(format_document(fields))
    classifications[filename] = result
    print(f"{filename}: siniflar={result['siniflar']}, guven={result['guven']}, human_review={result['human_review']}")

test_talep_01.png: siniflar=['talep formu'], guven=0.95, human_review=False


test_talep_02.png: siniflar=['talep formu'], guven=0.9, human_review=False


test_talep_03.png: siniflar=['talep formu'], guven=0.85, human_review=False


test_talep_04.png: siniflar=['talep formu'], guven=0.9, human_review=False


test_talep_05.png: siniflar=['talep formu'], guven=0.9, human_review=False


## 3. Belge bazlı chunklama (DOC-23 ile aynı ayarlar)

`chunk_size=150`, `chunk_overlap=20` — DOC-23'te doğrulanan, 5 belgenin de bölünmeden tek chunk olarak kaldığı ayarlar. Her chunk `source_doc` alanıyla kaynağına bağlanıyor.

In [4]:
CHUNK_SIZE = 150
CHUNK_OVERLAP = 20

all_chunks = []
gid = 0
for filename in sorted(ocr_outputs.keys()):
    fields = ocr_outputs[filename]
    text = format_document(fields)
    doc_chunks = split_text(text, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    for c in doc_chunks:
        c["chunk_id"] = gid
        c["source_doc"] = filename
        c["konu"] = fields.get("konu")
        c["talep_eden"] = fields.get("talep_eden")
        gid += 1
    all_chunks.extend(doc_chunks)

print(f"{len(ocr_outputs)} belgeden toplam {len(all_chunks)} chunk uretildi.")

5 belgeden toplam 5 chunk uretildi.


## 4. Sınıflandırma meta verisini chunk'lara ekleme (DOC-25'in çekirdeği)

`attach_labels_to_chunks()`, her chunk'ın `source_doc` alanına bakarak ilgili belgenin sınıflandırma sonucunu (`siniflar`, `guven`, `etiketler`, `human_review`) chunk'ın kendisine kopyalıyor. `vector_store.build_index()` embedding dışındaki tüm alanları metadata'ya taşıdığı için, bu alanlar arama sonuçlarında da doğrudan görünecek.

In [5]:
labeled_chunks = attach_labels_to_chunks(all_chunks, classifications)

for c in labeled_chunks:
    assert "siniflar" in c and "guven" in c and "etiketler" in c and "human_review" in c

print("OK - tum chunk'lara siniflandirma meta verisi eklendi.")
print(json.dumps({k: labeled_chunks[0][k] for k in ["source_doc", "siniflar", "guven", "etiketler", "human_review"]}, ensure_ascii=False, indent=2))

OK - tum chunk'lara siniflandirma meta verisi eklendi.
{
  "source_doc": "test_talep_01.png",
  "siniflar": [
    "talep formu"
  ],
  "guven": 0.95,
  "etiketler": [
    "ekipman talebi",
    "monitör",
    "yazılım geliştirme",
    "donanım talebi"
  ],
  "human_review": false
}


## 5. Embedding üret ve FAISS index'ini etiketli metadata ile yeniden kur

Üretim artefaktları (`chunk_embeddings.json`, `faiss_index.faiss/.meta.json`) sınıflandırma meta verisi eklenmiş haliyle güncelleniyor.

In [6]:
embedded_chunks = embed_chunks(labeled_chunks)

CHUNKS_OUT = "../data/processed/chunk_embeddings.json"
with open(CHUNKS_OUT, "w", encoding="utf-8") as f:
    json.dump(embedded_chunks, f, ensure_ascii=False, indent=2)

index, metadata = build_index(embedded_chunks)
INDEX_PATH = os.path.join("..", load_index_path())
save_index(index, metadata, INDEX_PATH)

loaded_index, loaded_metadata = load_index(INDEX_PATH)
assert loaded_index.ntotal == index.ntotal
assert loaded_metadata == metadata
assert all("siniflar" in m and "human_review" in m for m in loaded_metadata)
print(f"OK - FAISS index kaydedildi ve dogrulandi -> {INDEX_PATH}.faiss ({loaded_index.ntotal} vektor), metadata siniflandirma alanlarini iceriyor.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

OK - FAISS index kaydedildi ve dogrulandi -> ..\./data/processed/faiss_index.faiss (5 vektor), metadata siniflandirma alanlarini iceriyor.


## 6. Uçtan uca zincir doğrulaması: zincir kırılmadan çalışıyor mu?

DOC-23'teki aynı 10 sorgu (5 belge x doğal+anlamsal) yeniden çalıştırılıyor. Amaç: sınıflandırma meta verisi eklenmesinin arama doğruluğunu bozmadığını (Hit@3 hâlâ %100) ve artık her sonucun kendi `siniflar`/`etiketler`/`human_review` bilgisini taşıdığını göstermek.

In [7]:
TEST_QUERIES = [
    {"query": "Ek monitor talebi olan kim?", "expected_doc": "test_talep_01.png"},
    {"query": "Log takibini kod yazimiyla eszamanli yurutmek isteyen kisi kim?", "expected_doc": "test_talep_01.png"},
    {"query": "Laptop talebinde bulunan kim?", "expected_doc": "test_talep_02.png"},
    {"query": "Cihazinin performansi dustugu icin verimliligi etkilenen calisan kim?", "expected_doc": "test_talep_02.png"},
    {"query": "Klavye degisikligi isteyen kim?", "expected_doc": "test_talep_03.png"},
    {"query": "Yazi yazarken zorlanan, donanimi arizali olan kisi kim?", "expected_doc": "test_talep_03.png"},
    {"query": "Yazici arizasi bildiren kim?", "expected_doc": "test_talep_04.png"},
    {"query": "Muhasebe departmaninda ortak kullanilan bir cihaz bozulmus, kim talep acti?", "expected_doc": "test_talep_04.png"},
    {"query": "Ek ekran talebi olan kim?", "expected_doc": "test_talep_05.png"},
    {"query": "Tasarim verimliligini artirmak icin ikinci ekrana ihtiyac duyan kisi kim?", "expected_doc": "test_talep_05.png"},
]

rows = []
for tq in TEST_QUERIES:
    query_embedding = embed_chunks([{"chunk_id": -1, "text": tq["query"], "token_count": 0, "source_doc": None}])[0]["embedding"]
    results = search(loaded_index, loaded_metadata, query_embedding, top_k=3)
    retrieved_docs = [r["source_doc"] for r in results]
    rank = retrieved_docs.index(tq["expected_doc"]) + 1 if tq["expected_doc"] in retrieved_docs else None
    top1 = results[0]
    rows.append({
        "sorgu": tq["query"],
        "beklenen_belge": tq["expected_doc"],
        "top1_belge": top1["source_doc"],
        "top1_siniflar": top1["siniflar"],
        "top1_human_review": top1["human_review"],
        "hit@1": rank == 1,
        "hit@3": rank is not None,
        "rank": rank,
    })

hit1 = sum(r["hit@1"] for r in rows) / len(rows)
hit3 = sum(r["hit@3"] for r in rows) / len(rows)

print(f"Hit@1={hit1:.0%}  Hit@3={hit3:.0%}  (n={len(rows)} sorgu)\n")
for r in rows:
    print(f"[{'OK' if r['hit@1'] else 'ranked ' + str(r['rank'])}] {r['sorgu'][:50]:50s} -> {r['top1_belge']} | siniflar={r['top1_siniflar']} human_review={r['top1_human_review']}")

assert hit3 == 1.0, "Sinifladirma meta verisi eklendikten sonra Hit@3 dustu - zincir kirilmis olabilir."
assert hit1 >= 0.7, "Hit@1 DOC-23'teki referans degerin (0.70) altina dustu."
print("\nOK - zincir kirilmadan calisiyor (Hit@3=%100 korundu) ve her sonuc kendi siniflandirma meta verisini tasiyor.")

Hit@1=80%  Hit@3=100%  (n=10 sorgu)

[ranked 3] Ek monitor talebi olan kim?                        -> test_talep_05.png | siniflar=['talep formu'] human_review=False
[OK] Log takibini kod yazimiyla eszamanli yurutmek iste -> test_talep_01.png | siniflar=['talep formu'] human_review=False
[OK] Laptop talebinde bulunan kim?                      -> test_talep_02.png | siniflar=['talep formu'] human_review=False
[OK] Cihazinin performansi dustugu icin verimliligi etk -> test_talep_02.png | siniflar=['talep formu'] human_review=False
[OK] Klavye degisikligi isteyen kim?                    -> test_talep_03.png | siniflar=['talep formu'] human_review=False
[OK] Yazi yazarken zorlanan, donanimi arizali olan kisi -> test_talep_03.png | siniflar=['talep formu'] human_review=False
[OK] Yazici arizasi bildiren kim?                       -> test_talep_04.png | siniflar=['talep formu'] human_review=False
[ranked 3] Muhasebe departmaninda ortak kullanilan bir cihaz  -> test_talep_01.png | siniflar=['

## 7. human_review meta verisiyle sonuç filtreleme örneği

Mentörün önerdiği gibi, düşük güvenli belgeler ayrı bir "belirsiz" sınıfına atılmak yerine `human_review=True` ile işaretleniyor ama indeksten çıkarılmıyor. Aşağıda arama sonuçlarının, istenirse `human_review=False` (yani insan onayından geçmiş / yeterince güvenilir) olanlarla nasıl filtrelenebileceği gösteriliyor — belge arama sonucundan hiç düşmüyor, sadece ek bir sinyal olarak kullanılabiliyor.

In [8]:
query_embedding = embed_chunks([{"chunk_id": -1, "text": "Ekipman talebi olan belgeler", "token_count": 0, "source_doc": None}])[0]["embedding"]
all_results = search(loaded_index, loaded_metadata, query_embedding, top_k=5)
reviewed_only = [r for r in all_results if not r["human_review"]]

print(f"Tum sonuclar: {len(all_results)} | human_review=False (guvenilir) olanlar: {len(reviewed_only)}")
for r in all_results:
    flag = "[İNCELEME BEKLİYOR]" if r["human_review"] else "[ONAYLI]"
    print(f"{flag} {r['source_doc']} | guven={r['guven']} | siniflar={r['siniflar']}")

Tum sonuclar: 5 | human_review=False (guvenilir) olanlar: 5
[ONAYLI] test_talep_01.png | guven=0.95 | siniflar=['talep formu']
[ONAYLI] test_talep_03.png | guven=0.85 | siniflar=['talep formu']
[ONAYLI] test_talep_04.png | guven=0.9 | siniflar=['talep formu']
[ONAYLI] test_talep_05.png | guven=0.9 | siniflar=['talep formu']
[ONAYLI] test_talep_02.png | guven=0.9 | siniflar=['talep formu']


## 8. İnsan incelemesi sonrası metadata güncelleme (index'i yeniden kurmadan)

Mentörün notu: *"İnsan incelemesi sonrasında ise ilgili kategori ve metadata bilgileri güncellenebilir."* Şu ana kadarki akış, `human_review=True` olan bir belgeyi indeksten düşürmeden aranabilir tutuyordu; bu bölüm, bir insan incelemecinin o belgeyi onayladıktan/düzelttikten **sonra** bu düzeltmeyi nasıl kalıcı hale getirdiğimizi gösteriyor.

`vector_store.update_metadata_by_source_doc()` + `save_metadata()`, tek bir belgenin `source_doc` alanına göre metadata'sını günceller ve sadece `.meta.json` dosyasını yeniden yazar — vektörler değişmediği için FAISS `.faiss` dosyası (ve dolayısıyla `build_index`/`save_index` çağrısı) hiç dokunulmadan kalır. Yani düzeltme, tüm pipeline'ı (embedding + index) yeniden kurmayı gerektirmiyor.

In [9]:
from vector_store import update_metadata_by_source_doc, save_metadata

REVIEW_TARGET = "test_talep_03.png"
faiss_bytes_before = open(INDEX_PATH + ".faiss", "rb").read()

human_correction = {
    "etiketler": loaded_metadata[[m["source_doc"] for m in loaded_metadata].index(REVIEW_TARGET)]["etiketler"] + ["acil"],
    "incelendi": True,
    "inceleyen": "Dila Alpay",
    "inceleme_tarihi": "2026-08-22",
}

updated_metadata = update_metadata_by_source_doc(loaded_metadata, REVIEW_TARGET, human_correction)
save_metadata(updated_metadata, INDEX_PATH)

reloaded_index, reloaded_metadata = load_index(INDEX_PATH)
faiss_bytes_after = open(INDEX_PATH + ".faiss", "rb").read()

updated_entry = next(m for m in reloaded_metadata if m["source_doc"] == REVIEW_TARGET)
untouched_entry = next(m for m in reloaded_metadata if m["source_doc"] == "test_talep_01.png")

assert updated_entry["incelendi"] is True
assert "acil" in updated_entry["etiketler"]
assert "incelendi" not in untouched_entry, "Guncelleme baska bir belgeye sizmis olmamali."
assert faiss_bytes_after == faiss_bytes_before, "FAISS .faiss dosyasi degismemis olmali (sadece metadata guncellendi)."
assert reloaded_index.ntotal == loaded_index.ntotal

print(f"OK - '{REVIEW_TARGET}' metadata'si FAISS index'i yeniden kurmadan guncellendi:")
print(json.dumps({k: updated_entry[k] for k in ["source_doc", "siniflar", "etiketler", "incelendi", "inceleyen"]}, ensure_ascii=False, indent=2))
print(f"\nOK - FAISS .faiss dosyasi bayt-bayt ayni kaldi (vektorlere dokunulmadi), diger belgelerin metadata'si etkilenmedi.")

OK - 'test_talep_03.png' metadata'si FAISS index'i yeniden kurmadan guncellendi:
{
  "source_doc": "test_talep_03.png",
  "siniflar": [
    "talep formu"
  ],
  "etiketler": [
    "klavye değişimi",
    "arıza bildirimi",
    "ekipman talebi",
    "it talebi",
    "acil"
  ],
  "incelendi": true,
  "inceleyen": "Dila Alpay"
}

OK - FAISS .faiss dosyasi bayt-bayt ayni kaldi (vektorlere dokunulmadi), diger belgelerin metadata'si etkilenmedi.


## Sonuç

DOC-25 tamamlandı: `classify_document()` çıktıları (`siniflar`, `guven`, `etiketler`, `human_review`) `attach_labels_to_chunks()` ile her chunk'a meta veri olarak eklendi ve üretim FAISS index'ine (`data/processed/chunk_embeddings.json`, `faiss_index.faiss/.meta.json`) yazıldı.

**Uçtan uca zincir #2 (OCR → RAG → Sınıflandırma)** doğrulandı: aynı 5 test belgesi ve DOC-23'teki 10 sorgu üzerinden Hit@3=%100 korundu, ayrıca her arama sonucu artık kendi sınıf/etiket/human_review bilgisini taşıyor — böylece arama sonuçları kategoriye veya inceleme durumuna göre filtrelenebilir hale geldi.

Ek olarak, `vector_store.update_metadata_by_source_doc()` + `save_metadata()` ile insan incelemesi sonrası kategori/etiket düzeltmelerinin, FAISS index'i (vektörleri) yeniden kurmadan sadece metadata dosyasını güncelleyerek kalıcı hale getirilebildiği gösterildi — mentörün "inceleme sonrası kategori/metadata güncellenebilir" notu artık somut bir fonksiyonla karşılanıyor.